# Teil 3 - Modell

In diesem Notebook trainiere ich ein Modell, das CO2-Emissionen vorhersagen soll. Ich verwende dafür die Felder `year` und `energy_per_capita`. Daraus soll das Modell den Wert `co2` ableiten. Das Modell kennt die Zusammenhänge am Anfang nicht. Es lernt sie zuerst mit `fit()` aus den Trainingsdaten und macht danach mit `predict()` Vorhersagen für neue Testdaten.

In [1]:
import csv
import math
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

features = ["year", "energy_per_capita"]
target = "co2"

with open("data/co2_clean_1000.csv", newline="", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

X = np.array([[float(row[column]) for column in features] for row in rows], dtype=float)
y = np.array([float(row[target]) for row in rows], dtype=float)
row_indices = np.arange(len(rows))

print(f"Zeilen: {len(rows)}")
print("Eingabefelder: " + ", ".join(features))
print("Zielfeld: " + target)

Zeilen: 1000
Eingabefelder: year, energy_per_capita
Zielfeld: co2


## 3.1 Aufteilung in Trainings- und Testdaten

Ich teile den Datensatz mit `train_test_split` auf. 80 Prozent der Zeilen werden Trainingsdaten, 20 Prozent bleiben Testdaten. Diese Trennung ist wichtig, weil das Modell die Testdaten beim Lernen nicht sieht. Wenn es dort trotzdem brauchbare Vorhersagen macht, hat es eher einen echten Zusammenhang gelernt und nicht nur bekannte Werte auswendig übernommen.

In [2]:
X_train, X_test, y_train, y_test, train_indices, test_indices = train_test_split(
    X,
    y,
    row_indices,
    test_size=0.2,
    random_state=42,
)

print(f"Trainingsdaten: {len(X_train)}")
print(f"Testdaten: {len(X_test)}")

Trainingsdaten: 800
Testdaten: 200


## 3.2 Algorithmus und trainiertes Modell

Ich vergleiche zwei sklearn-Varianten: lineare Regression und Entscheidungsbaum. Die lineare Regression ist einfach, erkennt hier aber zu wenig, weil der Zusammenhang nicht sauber gerade ist. Der Entscheidungsbaum kann Schwellenwerte bilden, zum Beispiel beim Energieverbrauch. Ich begrenze ihn auf Tiefe 2, damit er verständlich bleibt und nicht zu stark auswendig lernt. Deshalb verwende ich den Entscheidungsbaum als finales Modell.

In [3]:
candidates = {
    "Lineare Regression": Pipeline(steps=[
        ("scaler", StandardScaler()),
        ("regression", LinearRegression()),
    ]),
    "Entscheidungsbaum": DecisionTreeRegressor(
        max_depth=2,
        random_state=42,
    ),
}

print("Vergleich der Modellvarianten")
print("Modell                 R2      RMSE     MAE")
for name, candidate in candidates.items():
    candidate.fit(X_train, y_train)
    candidate_predictions = candidate.predict(X_test)
    mse = mean_squared_error(y_test, candidate_predictions)
    rmse = math.sqrt(mse)
    mae = mean_absolute_error(y_test, candidate_predictions)
    r2 = r2_score(y_test, candidate_predictions)
    print(f"{name:<22} {r2:6.3f} {rmse:8.3f} {mae:7.3f}")

model = candidates["Entscheidungsbaum"]
predictions = model.predict(X_test)

print()
print("Finales Modell: Entscheidungsbaum")
print("Das Modell hat mit fit() aus den Trainingsdaten gelernt.")
print(f"Baumtiefe: {model.get_depth()}")
print(f"Blätter: {model.get_n_leaves()}")

Vergleich der Modellvarianten
Modell                 R2      RMSE     MAE
Lineare Regression      0.064  109.009  72.345
Entscheidungsbaum       0.270   96.291  57.306

Finales Modell: Entscheidungsbaum
Das Modell hat mit fit() aus den Trainingsdaten gelernt.
Baumtiefe: 2
Blätter: 4


## 3.3 Vorhersagen aus dem Testdatensatz

Nach dem Training lasse ich das Modell Vorhersagen für Testdaten machen. In der Tabelle stehen der echte CO2-Wert, die Vorhersage und die Abweichung. So sieht man direkt, wo das Modell ungefähr richtig liegt und wo es daneben liegt. Gerade diese Fehler sind nützlich, weil sie zeigen, welche Zusammenhänge noch fehlen.

In [4]:
print("Land         Jahr  echt      vorhergesagt  Abweichung")
for row_number, data_index in enumerate(test_indices[:8]):
    row = rows[int(data_index)]
    actual = y_test[row_number]
    predicted = predictions[row_number]
    difference = predicted - actual
    print(f"{row['country']:<12} {row['year']} {actual:9.3f} {predicted:13.3f} {difference:11.3f}")

Land         Jahr  echt      vorhergesagt  Abweichung
Bangladesh   2013    64.763        11.480     -53.283
Bolivia      2004    11.051        11.480       0.429
Bolivia      2007    12.740        11.480      -1.260
Belgium      2013   102.816       141.215      38.399
Azerbaijan   2004    32.253        85.985      53.732
Benin        1988     0.508        11.480      10.972
Belgium      1979   139.787       141.215       1.428
Bangladesh   2005    37.717        11.480     -26.237


Die manuelle Prüfung zeigt ein gemischtes, aber nachvollziehbares Bild. Der Entscheidungsbaum erkennt viele Fälle richtig als eher hohe oder niedrige Emissionen. Die genauen CO2-Werte sind aber nicht immer nahe am echten Wert, weil der Baum mit Tiefe 2 nur wenige grobe Gruppen bildet. Für eine einfache erste Lösung ist das brauchbar, aber nicht perfekt.